# Snowflake Arctic Embed S + Logistic Regression

Dùng `Snowflake/snowflake-arctic-embed-s` pretrained để embedding review, sau đó huấn luyện hai Logistic Regression. Chỉ đọc Train và Validation hiện tại; không chia lại, không thay đổi dữ liệu, không fine-tune Transformer và không mở/đánh giá trên Test. Model card khuyến nghị prompt `query`, được dùng khi mã hóa review.

In [1]:
from pathlib import Path
from time import perf_counter
import os

import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, precision_score, recall_score,
)

DATA_DIR = Path('data/processed')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/processed')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
validation_df = pd.read_csv(DATA_DIR / 'validation.csv')

# Sanity checks: use the existing CSVs as-is; do not resplit or write them.
assert len(train_df) == 5562
assert len(validation_df) == 1192
assert list(train_df.columns) == ['review', 'label']
assert list(validation_df.columns) == ['review', 'label']
assert train_df['review'].notna().all() and validation_df['review'].notna().all()
assert set(train_df['label'].unique()).issubset({0, 1})
assert set(validation_df['label'].unique()).issubset({0, 1})

def review_key(text):
    return str(text).casefold().strip()
assert not (set(train_df['review'].map(review_key)) &
            set(validation_df['review'].map(review_key)))

print(f'Train: {len(train_df):,} dòng; Validation: {len(validation_df):,} dòng')
print('Tỷ lệ nhãn Train:', train_df['label'].value_counts(normalize=True).sort_index().round(4).to_dict())
print('Tỷ lệ nhãn Validation:', validation_df['label'].value_counts(normalize=True).sort_index().round(4).to_dict())
print('Tập dữ liệu trùng review giữa Train và Validation: không')

Train: 5,562 dòng; Validation: 1,192 dòng
Tỷ lệ nhãn Train: {0: 0.0708, 1: 0.9292}
Tỷ lệ nhãn Validation: {0: 0.0705, 1: 0.9295}
Tập dữ liệu trùng review giữa Train và Validation: không


## 1. Tạo embedding bằng Snowflake Arctic Embed S
Mô hình pretrained chạy trên CPU. `prompt_name='query'` sử dụng hướng dẫn truy vấn được cung cấp cùng model. Ghi nhận riêng thời gian nạp mô hình và thời gian mã hóa Train/Validation; không tạo embedding cho Test.

In [2]:
torch.set_num_threads(min(4, os.cpu_count() or 1))
model_name = 'Snowflake/snowflake-arctic-embed-s'
load_start = perf_counter()
embedding_model = SentenceTransformer(model_name, device='cpu')
model_load_seconds = perf_counter() - load_start

train_start = perf_counter()
X_train_embeddings = embedding_model.encode(
    train_df['review'].tolist(), batch_size=32, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=False, prompt_name='query',
)
train_embedding_seconds = perf_counter() - train_start

validation_start = perf_counter()
X_validation_embeddings = embedding_model.encode(
    validation_df['review'].tolist(), batch_size=32, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=False, prompt_name='query',
)
validation_embedding_seconds = perf_counter() - validation_start

y_train = train_df['label'].to_numpy()
y_validation = validation_df['label'].to_numpy()
print(f'Thiết bị: CPU | Số chiều embedding: {X_train_embeddings.shape[1]}')
print(f'Thời gian nạp mô hình: {model_load_seconds:.2f} giây')
print(f'Thời gian embedding Train: {train_embedding_seconds:.2f} giây')
print(f'Thời gian embedding Validation: {validation_embedding_seconds:.2f} giây')
print('Kích thước ma trận:', X_train_embeddings.shape, X_validation_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ADMIN\Documents\Elden-Ring-Sentiment-Analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--Snowflake--snowflake-arctic-embed-s. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/85.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/703 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Thiết bị: CPU | Số chiều embedding: 384
Thời gian nạp mô hình: 24.57 giây
Thời gian embedding Train: 54.25 giây
Thời gian embedding Validation: 12.85 giây
Kích thước ma trận: (5562, 384) (1192, 384)


## 2. Logistic Regression và đánh giá Validation
So sánh cấu hình mặc định với `class_weight='balanced'`. Báo cáo Precision/Recall/F1 của lớp 1, Macro-F1, các chỉ số riêng của lớp 0, thời gian `fit` và ma trận nhầm lẫn (nhãn 0 trước nhãn 1).

In [3]:
model_configs = {
    'Mặc định': LogisticRegression(max_iter=1000),
    'class_weight=balanced': LogisticRegression(max_iter=1000, class_weight='balanced'),
}
results = []
confusion_matrices = {}
trained_models = {}

for name, classifier in model_configs.items():
    start = perf_counter()
    classifier.fit(X_train_embeddings, y_train)
    training_seconds = perf_counter() - start
    predictions = classifier.predict(X_validation_embeddings)
    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(
        y_validation, predictions, labels=[0, 1], zero_division=0
    )
    results.append({
        'Cấu hình': name,
        'Accuracy': accuracy_score(y_validation, predictions),
        'Precision (lớp 1)': precision_score(y_validation, predictions, pos_label=1, zero_division=0),
        'Recall (lớp 1)': recall_score(y_validation, predictions, pos_label=1, zero_division=0),
        'F1 (lớp 1)': f1_score(y_validation, predictions, pos_label=1, zero_division=0),
        'Macro-F1': f1_score(y_validation, predictions, average='macro', zero_division=0),
        'Precision (lớp 0)': class_precision[0],
        'Recall (lớp 0)': class_recall[0],
        'F1 (lớp 0)': class_f1[0],
        'Thời gian train (giây)': training_seconds,
    })
    confusion_matrices[name] = confusion_matrix(y_validation, predictions, labels=[0, 1])
    trained_models[name] = classifier

results_table = pd.DataFrame(results).set_index('Cấu hình')
display(results_table.round(4))
for name, matrix in confusion_matrices.items():
    print(f'Ma trận nhầm lẫn — {name} (hàng = nhãn thật, cột = dự đoán):')
    display(pd.DataFrame(matrix, index=['Thật 0', 'Thật 1'],
                         columns=['Dự đoán 0', 'Dự đoán 1']))

,Accuracy,Precision (lớp 1),Recall (lớp 1),F1 (lớp 1),Macro-F1,Precision (lớp 0),Recall (lớp 0),F1 (lớp 0),Thời gian train (giây)
Cấu hình,,,,,,,,,
Mặc định,0.9413,0.9421,0.9982,0.9693,0.6415,0.8889,0.1905,0.3137,0.0307
class_weight=balanced,0.8691,0.9818,0.8755,0.9256,0.6920,0.3235,0.7857,0.4583,0.0171


Ma trận nhầm lẫn — Mặc định (hàng = nhãn thật, cột = dự đoán):


,Dự đoán 0,Dự đoán 1
Thật 0,16,68
Thật 1,2,1106


Ma trận nhầm lẫn — class_weight=balanced (hàng = nhãn thật, cột = dự đoán):


,Dự đoán 0,Dự đoán 1
Thật 0,66,18
Thật 1,138,970


## 3. So sánh với các phương pháp đã chạy
Các kết quả TF-IDF và MiniLM là metric Validation đã lưu trong notebook 03 và 04. So sánh theo cùng cấu hình; không có kết quả Test trong bảng.

In [4]:
previous_results = {
    ('TF-IDF', 'Mặc định'): {'Accuracy': 0.9304, 'Macro-F1': 0.5049, 'F1 lớp 0': 0.0460},
    ('TF-IDF', 'class_weight=balanced'): {'Accuracy': 0.8842, 'Macro-F1': 0.6827, 'F1 lớp 0': 0.4298},
    ('MiniLM', 'Mặc định'): {'Accuracy': 0.9421, 'Macro-F1': 0.6433, 'F1 lớp 0': 0.3168},
    ('MiniLM', 'class_weight=balanced'): {'Accuracy': 0.8423, 'Macro-F1': 0.6589, 'F1 lớp 0': 0.4088},
}
comparison_rows = [
    {'Phương pháp': method, 'Cấu hình': config, **scores}
    for (method, config), scores in previous_results.items()
]
for config, row in results_table.iterrows():
    comparison_rows.append({
        'Phương pháp': 'Snowflake Arctic Embed S', 'Cấu hình': config,
        'Accuracy': row['Accuracy'], 'Macro-F1': row['Macro-F1'],
        'F1 lớp 0': row['F1 (lớp 0)'],
    })
comparison = pd.DataFrame(comparison_rows).set_index(['Phương pháp', 'Cấu hình'])
display(comparison.round(4))
best_index = comparison['Macro-F1'].idxmax()
print(f'Macro-F1 cao nhất: {best_index[0]} / {best_index[1]} ' +
      f"({comparison.loc[best_index, 'Macro-F1']:.4f})")
print('Chỉ sử dụng Validation; Test không được đọc hoặc đánh giá trong notebook này.')

Accuracy  Macro-F1  F1 lớp 0
Phương pháp              Cấu hình                                           
TF-IDF                   Mặc định                 0.9304    0.5049    0.0460
                         class_weight=balanced    0.8842    0.6827    0.4298
MiniLM                   Mặc định                 0.9421    0.6433    0.3168
                         class_weight=balanced    0.8423    0.6589    0.4088
Snowflake Arctic Embed S Mặc định                 0.9413    0.6415    0.3137
                         class_weight=balanced    0.8691    0.6920    0.4583

Macro-F1 cao nhất: Snowflake Arctic Embed S / class_weight=balanced (0.6920)
Chỉ sử dụng Validation; Test không được đọc hoặc đánh giá trong notebook này.
